In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document  # 确保导入了 Document

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

folder_path = "../data/"
all_docs = [] 

print(f"开始扫描 {folder_path} 目录下的txt文件")

# 1. 遍历文件夹，读取文件
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        print(f"   -> 发现文件: {filename}，正在读取...")
        
        # 准备不同的文件编码格式
        keys = ["utf-8", "utf-8-sig", "gb18030", "gbk", "gb2312"]
        success = False
        
        for key in keys:
            try:
                loader = TextLoader(file_path, encoding=key)
                docs = loader.load() # 得到 Document 对象列表
                all_docs.extend(docs) # 将当前文件的 Document 对象添加到总列表中
                
                print(f" 成功使用 [{key}] 编码读取")
                success = True
                break # 成功了就不试其他编码了，直接跳出循环
            except Exception:
                continue # 如果当前编码失败了，就继续试下一个编码
                
        # 如果所有编码都失败了，就启动暴力推土机模式
        if not success:
            print(f"  警告：{filename} 编码混乱，启动暴力推土机模式...")
            # 用原生 Python 强行读取，过滤死字节
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                raw_text = f.read()
            # 直接把整个文本当成一个 Document 对象，只能入库，无法进行切分
            doc = Document(page_content=raw_text, metadata={"source": file_path})
            all_docs.append(doc)

print(f" 文件扫描完毕！一共读取了 {len(all_docs)} 份文档。")

# 2. 切分和入库 
print(" 正在进行统一文本切分...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(all_docs)
print(f" 成功！所有文件被切分成了 {len(chunks)} 个知识块。")

print(" 正在加载本地 Embedding 模型...")
embeddings = HuggingFaceEmbeddings()

print(f" 正在将 {len(chunks)} 个知识块转化为向量并写入 Chroma 数据库...")
db = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="../db")
print(" 所有文件批量入库彻底完成！")

In [ ]:
# 4. 模拟一次真实业务中的 RAG 检索
query = "什么是敏感个人信息？"
print(f"【测试问题】: {query}\n")

# 让 Chroma 数据库去寻找与 query 向量空间最接近的 2 个数据块 (k=5)
results = db.similarity_search(query, k=5) 

for i, res in enumerate(results):
    print(f"--- 💡 数据库检索到的第 {i+1} 条相关依据 ---")
    print(res.page_content)
    print("-" * 40)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
import os

load_dotenv("api_key.env", override=True) # 加载 .env 文件 同时重置环境变量

# 1. 唤醒大模型 
print("正在连接 DeepSeek 大脑...")
llm = ChatOpenAI(
    model="deepseek-chat", 
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    max_tokens=1024,
    temperature=0.1 # 设置为0.1让它回答更严谨，不要瞎发散
)

# 2. 把刚才检索到的 2 条法律依据拼成一段大白话
context_text = "\n".join([doc.page_content for doc in results])

# 3. 构建 CoT 提示词 展示模型思维链
prompt_template = f"""
你是一个顶级的数据安全合规专家。
请根据以下我为你检索到的《个人信息保护法》相关条文：
【参考依据开始】
{context_text}
【参考依据结束】

请对我提供的以下业务字段进行分类分级判定：
- 字段名称：user_health_record
- 业务描述：患者在医院的既往病史和生理健康检测记录

请严格按照以下步骤思考并输出结果：
Step 1: 语义拆解该字段代表的实体性质。
Step 2: 将其与参考依据中的定义进行比对。
Step 3: 给出最终的分类和分级结论（如 L1-L4）。
"""

# 4. 发送给大模型并打印结果
print("正在请大模型进行推理分析，请稍候...\n")
response = llm.invoke(prompt_template)

print("🌟 判定结果出炉：")
print(response.content)

In [ ]:
import pandas as pd
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. 加载环境变量并唤醒大模型
load_dotenv("api_key.env", override=True)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"), 
    max_tokens=1024,
    temperature=0.1 
)

# 2. 连接 Chroma 法律知识库
print("正在连接本地法律知识库...")
local_model_path = r"G:\AI_Models\sentence-transformer"

embeddings = HuggingFaceEmbeddings(
    model_name=local_model_path,
    model_kwargs={'local_files_only': True} 
)
db = Chroma(persist_directory="../db", embedding_function=embeddings)

# 3. 读取准备的测试数据
print("正在读取 Excel 测试数据...")

excel_path = "../testdata/财务与会计.xlsx" 

# Pandas 读取 Excel 文件
df = pd.read_excel(excel_path)

# 进行测试
test_df = df

# 创建一个空列表，用来装大模型的判定结果
results_list = []

# 4. 开启自动化引擎：遍历每一行数据
print(f" 开始批量测试，共计 {len(test_df)} 条数据...\n")

for index, row in test_df.iterrows():
    field_en = row['英文名']
    field_cn = row['中文名']
    desc = row['描述']
    
    print(f"👉 正在处理: {field_cn} ({field_en})...")
    
    #  A：去知识库捞法律依据
    query = f"关于 {field_cn} {desc} 的数据安全和保密级别规定"
    docs = db.similarity_search(query, k=2)
    context_text = "\n".join([doc.page_content for doc in docs])
    
    #  B：组装 CoT 思维链 Prompt
    prompt_template = f"""
    你是一个顶级的数据安全合规专家。
    请根据以下我为你检索到的法律条文或行业规范：
    【参考依据开始】
    {context_text}
    【参考依据结束】

    请对我提供的以下业务字段进行分类分级判定：
    - 字段英文名：{field_en}
    - 字段中文名：{field_cn}
    - 业务描述：{desc}

    请严格按照以下步骤思考：
    Step 1 (语义分析): 该字段在真实业务中代表什么实体。
    Step 2 (合规比对): 将其与参考依据中的定级标准进行比对,并指出参考依据中的法律条文或规范名称。
    Step 3 (输出结论): 给出最终的分类和分级结论（如：L1一般数据，L2内部数据，L3敏感数据，L4核心数据）。
    """
    
    # 环节 C：连接DeepSeek 
    response = llm.invoke(prompt_template)
    
    # 将结果保存到我们的列表中
    results_list.append({
        "英文名": field_en,
        "中文名": field_cn,
        "业务描述": desc,
        "大模型分级结论": response.content
    })

# 5. 导出最终成果
print("\n 批量测试完成！正在生成结果报告...")
result_df = pd.DataFrame(results_list)

# 将结果保存为一个新的 CSV 文件
result_df.to_csv("../testdata/测试结果报告.csv", index=False, encoding='utf-8-sig')
print("报告已生成，请在左侧目录查看 测试结果报告.csv！")

In [1]:
import gradio as gr
import pandas as pd
import os
import traceback  # 引入详细报错追踪库

# --- 两个关键库导入 ---
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI 
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, classification_report

# ---- RAG组件和大模型初始化 ----
print("正在加载本地知识库与模型，请稍候...")

# 1. 初始化 Embedding 模型
embeddings = HuggingFaceEmbeddings(
    model_name=r"G:\AI_Models\sentence-transformer", 
    model_kwargs={'local_files_only': True}
)

# 2. 连接本地的 Chroma 数据库
vector_db = Chroma(persist_directory="../db", embedding_function=embeddings)

# 3. 初始化 DeepSeek 大模型
load_dotenv("api_key.env", override=True)
llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"), 
    max_tokens=1024,
    temperature=0.1 
)

# 4. 定义 CoT 思维链 Prompt 模板
prompt_template = PromptTemplate(
    input_variables=["context", "field_en", "field_cn", "desc"],
    template="""
    你是一个数据合规专家。请根据以下法律条文：
    {context}
    
    分析以下业务字段：
    英文名：{field_en}
    中文名：{field_cn}
    描述：{desc}
    
    请严格按照 Step 1, Step 2, Step 3 进行分析，并在最后一行明确输出级别（如 L1, L2, L3, L4）。
    """
)
print(" 底层引擎准备就绪！")

# ----  核心功能逻辑函数 ----
ERROR_LOG_CACHE = {}  

# 功能 1：加载错题集
def load_error_log(file_obj):
    global ERROR_LOG_CACHE
    if file_obj is None: return " 未上传文件。"
    try:
        # 注意：这里的 'name' 和 '标准答案' 必须是表格里真实的列名！
        df = pd.read_excel(file_obj.name) if file_obj.name.endswith('.xlsx') else pd.read_csv(file_obj.name)
        ERROR_LOG_CACHE = dict(zip(df['name'], df['标准答案']))
        return f" 错题集加载成功！已激活 {len(ERROR_LOG_CACHE)} 条强制拦截规则。"
    except Exception as e:
        return f" 格式有误: {str(e)}"

# 功能 2：动态更新知识库（txt格式）
def update_knowledge_base(file_objs):
    if not file_objs: return " 未选择文件。"
    try:
        total_chunks = 0
        for file_obj in file_objs:
            with open(file_obj.name, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
            chunks = text_splitter.create_documents([text])
            vector_db.add_documents(chunks)
            total_chunks += len(chunks)
        vector_db.persist()
        return f" 知识库更新成功！新增 {total_chunks} 个知识区块并已持久化到本地。"
    except Exception as e:
        return f" 更新失败：{str(e)}"

# 功能 3：单条预测 (接入大模型)
def smart_predict(field_en, field_cn, desc):
    if field_en in ERROR_LOG_CACHE:
        return ERROR_LOG_CACHE[field_en], "🎯 [人工干预] 触发错题本强规则拦截，跳过大模型。", "⚠️ 命中强制拦截规则，无需检索法律依据。" 

    search_query = f"{field_cn} {desc}"
    docs = vector_db.similarity_search(search_query, k=3) # 从知识库中选取3条最相关的依据

    context_list = []
    for i, doc in enumerate(docs):
        context_list.append(f"【依据 {i+1}】: {doc.page_content}")
    retrieved_context_all = "\n\n".join(context_list)

    retrieved_context = "\n".join([doc.page_content for doc in docs])

    final_prompt = prompt_template.format(context=retrieved_context, field_en=field_en, field_cn=field_cn, desc=desc)

    try:
        # 调用大模型
        response = llm.invoke(final_prompt).content
        
        level = "未知"
        if "L4" in response: level = "L4"
        elif "L3" in response: level = "L3"
        elif "L2" in response: level = "L2"
        elif "L1" in response: level = "L1"
        
        return level, response, retrieved_context_all #返回级别和大模型的完整推理过程以及检索到的法律依据
    except Exception as e:
        return "Error", f" 调用大模型出错: {str(e)}"

# 功能 4：批量预测打分（debug）
def batch_evaluate(file_obj ,progress=gr.Progress()):
    try:
        # 定义log_content专门用来存实时日志
        log_content = " 开始执行批量评测任务...\n"
        # 使用yield把当前的日志推送给网页
        yield log_content, "处理中...", None

        # debug1 - 验证函数入口
        print("\n---  开始执行批量评测任务 ---")
        if file_obj is None: 
            return "请先上传测试集", None
            
        # debug2 - 验证文件格式
        print(f" 正在读取文件: {file_obj.name}")
        if file_obj.name.endswith('.xlsx') or file_obj.name.endswith('.xls'):
            df = pd.read_excel(file_obj.name)
        else:
            df = pd.read_csv(file_obj.name)
            
        # debug3 - 验证表格结构
        print(f" 读取成功！当前表格的列名有: {list(df.columns)}")
        required_columns = ['英文名', '中文名', '业务描述', '标准答案']
        for col in required_columns:
            if col not in df.columns:
                # 如果缺列或者列名不匹配，直接触发网页红色报错框
                raise gr.Error(f" 格式不对！找不到名为 '{col}' 的列，请检查 Excel 表头！")
                
        predictions, reasons, contexts = [], [], []

        log_content += f" 准备就绪，共计 {len(df)} 条数据，开始调用大模型...\n"
        log_content += "-" * 40 + "\n"
        yield log_content, "推理中...", None
        
        # debug4 - 验证循环逻辑
        print(f" 开始遍历 {len(df)} 条数据，准备调用大模型...")
        for index, row in progress.tqdm(df.iterrows(), total=len(df), desc="智能定级推演中"):
            print(f"   -> 正在推演第 {index + 1} 条数据: {row['英文名']}")
            lvl, rsn,ctx = smart_predict(row['英文名'], row['中文名'], row['业务描述']) 
            predictions.append(lvl)
            reasons.append(rsn)
            contexts.append(ctx)

            # 把当前这条的处理结果追加到日志里
            log_content += f" [{index + 1}/{len(df)}] 字段 '{row['英文名']}' -> 判定为: {lvl}\n"
            # 实时推送到网页的日志框里
            yield log_content, "推理中...", None
            
        log_content += "-" * 40 + "\n"
        log_content += " 大模型推理全部完成，正在计算准确率...\n"
        yield log_content, "结算中...", None
            
        # debug5 - 验证结果处理
        print(" 大模型推理全部完成，正在计算准确率...")
        df['大模型结论'] = predictions
        df['大模型理由'] = reasons
        df['检索到的法律依据'] = contexts

        report_dict = classification_report(df['标准答案'], df['大模型结论'], output_dict=True, zero_division=0)
        acc = accuracy_score(df['标准答案'], df['大模型结论'])
        
        l4_recall = report_dict.get('L4', {}).get('recall', 0)

        # debug6 - 验证结果导出(检索信息和错题本)
        print(" 正在生成多维度评测报告...")
        
        # 1. 准备错题集
        bad_cases_df = df[df['标准答案'] != df['大模型结论']]
        
        # 2. 将计算好的评估报告转化为 Pandas 表格，并翻译成中文表头
        metrics_df = pd.DataFrame(report_dict).transpose()
        metrics_df.reset_index(inplace=True)
        metrics_df.rename(columns={
            'index': '评测维度 (级别)', 
            'precision': '精确率 (Precision)', 
            'recall': '召回率 (Recall)', 
            'f1-score': 'F1 分数', 
            'support': '真实数据量'
        }, inplace=True)
        
        # 3. 定义导出路径
        report_path = "数据定级全量评测报告.xlsx"
        
        # 4. 使用 ExcelWriter 实现多 Sheet 导出
        with pd.ExcelWriter(report_path) as writer:
            # Sheet 1: 检索信息和大模型推理结果（全量数据）
            df.to_excel(writer, sheet_name='全量评测记录', index=False)
            # Sheet 2: 错题集（方便针对性指正）
            bad_cases_df.to_excel(writer, sheet_name='错题核查清单', index=False)
            # Sheet 3: 量化评测指标（整体准确率和各级别的精确率、召回率、F1分数）
            metrics_df.to_excel(writer, sheet_name='量化评估指标', index=False)
        
        log_content += "  报告生成完毕，包含全量数据与错题清单。\n"
        metrics_summary = f" 整体准确率: {acc*100:.2f}%\n L4 级别召回率: {l4_recall*100:.2f}%\n(请下载 Excel 查看详细检索依据)"
        
        # 返回文件路径
        yield log_content, metrics_summary, report_path

    # 捕获所有未知的致命错误
    except Exception as e:
        error_detail = traceback.format_exc()
        error_msg = f"\n================ ❌ 发现致命错误 ================\n{str(e)}\n"
        # 直接在 Gradio 网页右上角弹出一个红色的报错提示框！
        yield log_content + error_msg, "系统崩溃", None
        raise gr.Error(f"系统崩溃了！原因：{str(e)}")
    

#  ---- 网页 UI 搭建 ----
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ 智能合规定级引擎 (含人工干预与动态学习)")
    
    # "错题本"和"知识库更新" 并排放在网页顶部
    with gr.Row():
        with gr.Accordion("⚙️ 规则干预：上传错题本", open=False):
            error_file = gr.File(label="上传错题本 (.xlsx)")
            error_status = gr.Textbox(label="拦截器状态", value="未激活", interactive=False)
            error_file.upload(fn=load_error_log, inputs=error_file, outputs=error_status)
            
        with gr.Accordion("📚 知识扩充：上传最新法规", open=False):
            kb_files = gr.File(label="上传法规文本 (.txt)", file_count="multiple")
            kb_btn = gr.Button("🧠 吸收知识入库")
            kb_status = gr.Textbox(label="知识库状态", value="等待投喂...", interactive=False)
            kb_btn.click(fn=update_knowledge_base, inputs=kb_files, outputs=kb_status)

    # 主体功能区：单条测试和批量评测两个标签页
    with gr.Tabs():
        with gr.TabItem("🔍 单条语义测试"):
            gr.Markdown("输入单个业务字段，测试大模型的思维逻辑。")
            with gr.Row():
                with gr.Column():
                    input_en = gr.Textbox(label="字段英文名 (如: account_code)")
                    input_cn = gr.Textbox(label="字段中文名")
                    input_desc = gr.Textbox(label="业务描述", lines=3)
                    btn_single = gr.Button("🚀 运行单条判级", variant="primary")
                with gr.Column():
                    out_level = gr.Textbox(label="判定级别")
                    out_context = gr.Textbox(label="📖 检索到的法律依据 (Top 3)", lines=8)
                    out_reason = gr.Textbox(label="大模型推演理由", lines=5)
            btn_single.click(fn=smart_predict, inputs=[input_en, input_cn, input_desc], outputs=[out_level, out_reason, out_context])
            
        with gr.TabItem("📊 批量自动化评测"):
            gr.Markdown("上传包含 `标准答案` 的 Excel 测试集，系统将自动计算准确率并导出错题本。")
            with gr.Row():
                with gr.Column(scale=1):
                    test_file = gr.File(label="上传测试集 (.xlsx )")
                    btn_batch = gr.Button("🔥 启动批量评测", variant="primary")
                    
                # 🌟 新增的右侧“运行监控屏幕”
                with gr.Column(scale=2):
                    out_log = gr.Textbox(label=" 实时处理控制台", lines=12, max_lines=15, interactive=False)
            
            with gr.Row():
                out_acc = gr.Textbox(label="系统量化指标", lines=2)
                out_bad_cases = gr.File(label=" 下载错题诊断表")
            
            # 三个输出：实时日志、准确率文本、错题本下载链接
            btn_batch.click(fn=batch_evaluate, inputs=test_file, outputs=[out_log, out_acc, out_bad_cases],show_progress="hidden"  )
# 网页启动
demo.queue().launch(share=True, inbrowser=True)

正在加载本地知识库与模型，请稍候...


C:\Users\smf\AppData\Local\Temp\ipykernel_4552\60591756.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: G:\AI_Models\sentence-transformer
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\smf\AppData\Local\Temp\ipykernel_4552\60591756.py:26: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory="../db", embedding_function=embeddings)


 底层引擎准备就绪！


C:\Users\smf\AppData\Local\Temp\ipykernel_4552\60591756.py:226: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://db55bbcf4f545fa494.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
